In [ ]:
using Pkg

println(pwd())

Pkg.activate(".")
Pkg.develop(path="..")

using MiniAutoDiff

In [ ]:
a = Variable(5.)
b = Variable(3.)
c = Variable(2.)

function f(a, b, c)
    return a * exp(c*b) + c*c*a
end

MiniAutoDiff.diff(f(a, b, c))


delta = Variable(0.00001)
dfda= (f(a + delta, b, c).value - f(a, b, c).value)/delta.value
println("Autodiff df/da=", a.grad)
println("Numerical df/da=", dfda)

dfda= (f(a, b + delta, c).value - f(a, b, c).value)/delta.value
println("Autodiff df/db=", b.grad)
println("Numerical df/db=", dfda)


dfda= (f(a, b, c + delta).value - f(a, b, c).value)/delta.value
println("Autodiff df/dc=", c.grad)
println("Numerical df/dc=", dfda)



# Autodiff df/da=407.4287934927351
# Numerical df/da=407.4287934599851
# Autodiff df/db=4034.287934927351
# Numerical df/db=4034.32827808956
# Autodiff df/dc=6071.431902391027
# Numerical df/dc=6071.522724892019


In [ ]:
function res(f,v)
    
    f0 = f(v)
    MiniAutoDiff.diff(f0)

    dv = map(x->x.grad,v)

    sum(dv .* dv)

end

v = map(Variable, rand(2))

f(v) = sin(3*v[1]) * sin(2*v[2])

f0 = f(v)
path = [map(x->x.value,v)]

while res(f,v)>0.0001
    dv = map(x->x.grad,v)
    v = map(x->Variable(x.value), v .- 0.2 .* dv)
    push!(path, map(x->x.value,v))
end



In [ ]:
using CairoMakie

In [ ]:
vlim = 1.5*maximum(abs.(hcat(path...)))
x = -vlim:0.05:vlim
y = x
dmap = [f([xi,yi]) for xi in x, yi in y];

In [ ]:
fig = Figure()
ax = Axis(fig[1,1])

heatmap!(ax,x,y,dmap)
lines!(ax,hcat(path...)[1,:],hcat(path...)[2,:])
fig

In [ ]:
function test(v::Vector{DT}) where DT
    2*sin(4*v[1]-1)-1
end

model = Model([Linear(1,10),Tanh(),Linear(10,10),Tanh(),Linear(10,1)])

draw(model)

In [ ]:


trackLoss = Float64[]
trackEpoch = Float64[]

xPlot = 0:0.01:1
yTrue = map(x->test([x]), xPlot)

fig = Figure()
ax = Axis(fig[1,1])

lines!(ax,xPlot,yTrue; color = "red")

nepochs = 20
B = 5

for epoch in 1:nepochs
    x0 = rand(500)
    n_batches = div(length(x0), B)
    epoch_loss_sum = 0.0

    for batch_idx in 1:n_batches
        batch = x0[(batch_idx-1)*B + 1 : batch_idx*B]

        ŷ_batch = [forward(model, [x])[1] for x in batch]
        y_batch  = [test([x]) for x in batch]
        lossRes  = mse_loss(ŷ_batch, y_batch)

        epoch_loss_sum += lossRes.value
        push!(trackLoss, lossRes.value)

        MiniAutoDiff.diff(lossRes)
        update(model, GradientDescent(0.1))
    end

    push!(trackEpoch, epoch_loss_sum / n_batches)

    yModel = map(x->forward(model,[x])[1].value, xPlot)
    lines!(ax,xPlot,yModel; alpha=(epoch/nepochs)^2, color="blue")
end

fig


In [ ]:
fig = Figure()
ax = Axis(fig[1,1]; yscale=log)
plot!(ax, trackLoss)

fig

In [ ]:
fig = Figure()
ax = Axis(fig[1,1]; yscale=log)
plot!(ax, trackEpoch)

fig